In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from scipy.interpolate import RBFInterpolator, griddata, LinearNDInterpolator, SmoothBivariateSpline, SmoothSphereBivariateSpline, LSQBivariateSpline
from sklearn.cluster import KMeans
from sklearn import svm
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.metrics import r2_score
from typing import Sequence
from dataclasses import dataclass
from typing import NamedTuple
from typing import Mapping
from typing import TypeAlias
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from scipy import stats
#from pykrige.ok import OrdinaryKriging

sys.path.append('C:/Users/evils/PycharmProjects/approximationMap')

from control.experiment_data import ExperimentData

# Исходные данные

In [ ]:
ed = ExperimentData()
experiment_variations = ed.get_all_available_variations()
experiment_variations_with_CO = []

for experiment_variation in experiment_variations:
    if 'CO' in experiment_variation:
        experiment_variations_with_CO.append(experiment_variation)

experiment_data = {}

for experiment_variation in experiment_variations_with_CO:
    ed.get_experiment_data(*experiment_variation)
    experiment_data[*experiment_variation] = ed.df

experiment_data[('diesel', 'steam', 'CO')]

In [ ]:
def title_on_russian(title: str) -> str:
    titles_on_russian = {
        'diesel': 'Дизельное топливо',
        'crude_oil': 'Сырая нефть',
        'heavy_oil': 'Мазут',
        'kerosene': 'Керосин',
        'waste_oil': 'Отработанное масло',
        'waste_oil_40_2024': 'Отработанное масло 40С',
        'waste_oil_60_2024': 'Отработанное масло 60С',
        'd': ' ',
        'air': 'воздух',
        'steam': 'пар'
    }
    return titles_on_russian[title]

for experiment_name in experiment_data:
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    
    if 'air' in experiment_name:
        y = experiment_data[experiment_name].F_air
        ax.set_ylabel('Расход воздуха, кг/ч')
    else:
        y = experiment_data[experiment_name].F_steam
        ax.set_ylabel('Расход пара, кг/ч')
    
    x = experiment_data[experiment_name].F_fuel
    z = experiment_data[experiment_name].CO

    ax.scatter(x, y, z, c='red', marker='o', s=40, label='Опорные точки')

    ax.set_xlabel('Расход топлива, кг/ч')
    ax.set_zlabel('ppm', rotation=90, labelpad=-6)
    ax.set_title(
        f'{title_on_russian(experiment_name[0])}, разбавитель - {title_on_russian(experiment_name[1])}'
    )
    plt.legend()

    # Поворот графика
    ax.view_init(elev=15, azim=55)  # elev - угол возвышения, azim - азимут
    ax.tick_params(axis='z', which='major', pad=0, rotation=90)
    
    plt.tight_layout()
    plt.show()

# RBF

In [ ]:
def check_negative_values(Z):
    """Проверяет, содержит ли массив Z отрицательные значения."""
    return np.any(Z < 0)  # True, если есть хотя бы одно отрицательное значение

def count_negative_values(Z):
    """
    Считает количество и процент отрицательных значений в массиве Z.
    Возвращает кортеж: (количество_отрицательных, процент_отрицательных).
    """
    count = np.sum(Z < 0)
    total = Z.size
    percent = (count / total) * 100 if total != 0 else 0.0
    return count, round(percent, 2)  # Округляем до 2 знаков после запятой

rbf_data = {}

for experiment_name, data in experiment_data.items():
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    
    if 'air' in experiment_name:
        y = np.array(data.F_air)
        y_min, y_max = data.F_air.min(), data.F_air.max()
        f_additive =  data.F_air
        ax.set_ylabel("Расход воздуха, кг/ч")
    else:
        y = np.array(data.F_steam)
        y_min, y_max = data.F_steam.min(), data.F_steam.max()
        f_additive =  data.F_steam
        ax.set_ylabel("Расход пара, кг")
    
    y = y.reshape(-1, 1)
    x = np.array(data.F_fuel).reshape(-1, 1)
    x_min, x_max = data.F_fuel.min(), data.F_fuel.max()
    
    xy = np.concatenate((x, y), axis=1)
    
    z = np.array(data.CO)
    
    rbfi = RBFInterpolator(xy, z, kernel='linear')
        
    # Создаем плотную сетку
    x_grid = np.linspace(x_min, x_max, 100)
    y_grid = np.linspace(y_min, y_max, 100)
    X, Y = np.meshgrid(x_grid, y_grid)
    
    # Преобразуем сетку в формат (n_samples, 2)
    xy_grid = np.column_stack((X.ravel(), Y.ravel()))

    Z = rbfi(xy_grid)

    Z = Z.reshape(X.shape)

    # Заменяем отрицательные значения на нули
    Z = np.clip(Z, 0, None)

    if check_negative_values(Z):
        print(f'Отрицательные значения присутствуют в {experiment_name}')
        print(
            f'Количество отрицательных точек в поверхности: {count_negative_values(Z)[0]}, {count_negative_values(Z)[1]}%'
        )

    rbf_data[experiment_name] = X, Y, Z

    """ПОСТРОЕНИЕ ГРАФИКА"""
    # Поверхность интерполяции
    surf = ax.plot_surface(
        X, Y, Z,
        cmap="viridis",      # Цветовая карта
        alpha=0.8,           # Прозрачность
        antialiased=True     # Сглаживание
    )
    # Исходные точки данных
    ax.scatter(
        data.F_fuel, f_additive, data.CO,
        c="red", 
        s=20, 
        label="Исходные данные",
        depthshade=False
    )

    # Настройки графика
    ax.set_xlabel("Расход топлива")
    ax.set_zlabel('ppm', rotation=90, labelpad=-6)
    ax.set_title(
        f'{title_on_russian(experiment_name[0])}, разбавитель - {title_on_russian(experiment_name[1])}'
    )
    plt.legend()
    ax.view_init(elev=15, azim=55)  # elev - угол возвышения, azim - азимут
    ax.tick_params(axis='z', which='major', pad=-3, rotation=90)
    plt.tight_layout()
    plt.show()

# griddata

In [ ]:
def plot_grid_interpolation(method):
    """
    Строит 3D-графики интерполяции для экспериментальных данных с использованием griddata.
    
    Параметры:
      method : str, способ интерполяции ('linear', 'nearest', 'cubic')
    """
    grid_data = {}
    
    for experiment_name, data in experiment_data.items():
        fig = plt.figure()
        ax = fig.add_subplot(111, projection='3d')
        
        # Выбираем данные для оси y
        if 'air' in experiment_name:
            y = np.array(data.F_air)
            y_label = "Расход воздуха, кг/ч"
        else:
            y = np.array(data.F_steam)
            y_label = "Расход пара, кг"
        
        x = np.array(data.F_fuel)
        z = np.array(data.CO)
        
        # Определяем диапазоны для построения сетки
        x_min, x_max = x.min(), x.max()
        y_min, y_max = y.min(), y.max()
        
        # Создаем регулярную сетку
        x_grid = np.linspace(x_min, x_max, 100)
        y_grid = np.linspace(y_min, y_max, 100)
        X, Y = np.meshgrid(x_grid, y_grid)
        
        # Интерполяция с использованием griddata и заданного метода
        Z = griddata((x, y), z, (X, Y), method=method)

        if check_negative_values(Z):
            print(f'Отрицательные значения присутствуют в {experiment_name}')
            print(
            f'Количество отрицательных точек в поверхности: {count_negative_values(Z)[0]}, {count_negative_values(Z)[1]}%'
            )
        
        
        # Сохраняем результаты интерполяции
        grid_data[experiment_name] = (X, Y, Z)
        
        # Построение графика
        surf = ax.plot_surface(X, Y, Z, cmap="viridis", alpha=0.8, antialiased=True)
        ax.scatter(x, y, z, c="red", s=20, label="Исходные данные", depthshade=False)
        
        ax.set_xlabel("Расход топлива")
        ax.set_ylabel(y_label)
        ax.set_zlabel('ppm', rotation=90, labelpad=-6)
        ax.set_title(f'{title_on_russian(experiment_name[0])}, разбавитель - {title_on_russian(experiment_name[1])}')
        plt.legend()
        ax.view_init(elev=15, azim=55)
        ax.tick_params(axis='z', which='major', pad=-3, rotation=90)
        plt.tight_layout()
        plt.show()

In [ ]:
plot_grid_interpolation('linear')

In [ ]:
    plot_grid_interpolation('nearest')

In [ ]:
plot_grid_interpolation('cubic')

# LinearNDInterpolator

In [ ]:
linear_data = {}

for experiment_name, data in experiment_data.items():
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    
    # Выбираем данные для оси y
    if 'air' in experiment_name:
        y = np.array(data.F_air)
        y_label = "Расход воздуха, кг/ч"
    else:
        y = np.array(data.F_steam)
        y_label = "Расход пара, кг"
    
    x = np.array(data.F_fuel)
    z = np.array(data.CO)
    
    # Определяем диапазоны для построения сетки
    x_min, x_max = x.min(), x.max()
    y_min, y_max = y.min(), y.max()
    
    # Создаем регулярную сетку
    x_grid = np.linspace(x_min, x_max, 100)
    y_grid = np.linspace(y_min, y_max, 100)
    X, Y = np.meshgrid(x_grid, y_grid)
    
    # Подготавливаем исходные точки для интерполяции
    points = np.column_stack((x, y))
    
    # Создаем интерполятор LinearNDInterpolator
    interpolator = LinearNDInterpolator(points, z)
    
    # Выполняем интерполяцию на созданной сетке
    grid_points = np.column_stack((X.ravel(), Y.ravel()))
    Z = interpolator(grid_points).reshape(X.shape)

    if check_negative_values(Z):
        print(f'Отрицательные значения присутствуют в {experiment_name}')
        print(
            f'Количество отрицательных точек в поверхности: {count_negative_values(Z)[0]}, {count_negative_values(Z)[1]}%'
        )
    
    # Сохраняем результаты интерполяции
    linear_data[experiment_name] = (X, Y, Z)
    
    # Построение графика
    surf = ax.plot_surface(X, Y, Z, cmap="viridis", alpha=0.8, antialiased=True)
    ax.scatter(x, y, z, c="red", s=20, label="Исходные данные", depthshade=False)
    
    ax.set_xlabel("Расход топлива")
    ax.set_ylabel(y_label)
    ax.set_zlabel('ppm', rotation=90, labelpad=-6)
    ax.set_title(f'{title_on_russian(experiment_name[0])}, разбавитель - {title_on_russian(experiment_name[1])}')
    plt.legend()
    ax.view_init(elev=15, azim=55)
    ax.tick_params(axis='z', which='major', pad=-3, rotation=90)
    plt.tight_layout()
    plt.show()

# SmoothBivariateSpline

In [ ]:
spline_data = {}

for experiment_name, data in experiment_data.items():
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    
    # Выбираем данные для оси y
    if 'air' in experiment_name:
        y = np.array(data.F_air)
        y_label = "Расход воздуха, кг/ч"
    else:
        y = np.array(data.F_steam)
        y_label = "Расход пара, кг"
    
    x = np.array(data.F_fuel)
    z = np.array(data.CO)
    
    # Определяем диапазоны для построения сетки
    x_min, x_max = x.min(), x.max()
    y_min, y_max = y.min(), y.max()
    
    # Создаем регулярную сетку
    x_grid = np.linspace(x_min, x_max, 100)
    y_grid = np.linspace(y_min, y_max, 100)
    X, Y = np.meshgrid(x_grid, y_grid)

    s_value = 0.1 * len(x)
    # Создаем интерполятор SmoothBivariateSpline с параметром сглаживания s=0 для точного интерполирования
    spline = SmoothBivariateSpline(x, y, z, s=s_value)
    
    # Выполняем интерполяцию на созданной сетке
    # Метод ev() позволяет вычислить значения интерполированной функции в заданных точках
    Z = spline.ev(X.ravel(), Y.ravel()).reshape(X.shape)

     # Обрезаем значения выше 1100
    Z[Z > 1100] = 1100
    
    # Проверка на отрицательные значения
    if check_negative_values(Z):
        print(f'Отрицательные значения присутствуют в {experiment_name}')
        neg_count, neg_percent = count_negative_values(Z)
        print(f'Количество отрицательных точек в поверхности: {neg_count}, {neg_percent}%')
    
    # Сохраняем результаты интерполяции
    spline_data[experiment_name] = (X, Y, Z)
    
    # Построение графика
    surf = ax.plot_surface(X, Y, Z, cmap="viridis", alpha=0.8, antialiased=True)
    ax.scatter(x, y, z, c="red", s=20, label="Исходные данные", depthshade=False)
    
    ax.set_xlabel("Расход топлива")
    ax.set_ylabel(y_label)
    ax.set_zlabel("ppm", rotation=90, labelpad=-6)
    ax.set_title(f'{title_on_russian(experiment_name[0])}, разбавитель - {title_on_russian(experiment_name[1])}')
    plt.legend()
    ax.view_init(elev=15, azim=55)
    ax.tick_params(axis='z', which='major', pad=-3, rotation=90)
    plt.tight_layout()
    plt.show()

In [ ]:
spline_data = {}

for experiment_name, data in experiment_data.items():
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    
    # Выбираем данные для оси y
    if 'air' in experiment_name:
        y = np.array(data.F_air)
        y_label = "Расход воздуха, кг/ч"
    else:
        y = np.array(data.F_steam)
        y_label = "Расход пара, кг"
    
    x = np.array(data.F_fuel)
    z = np.array(data.CO)
    
    # Определяем диапазоны для построения сетки
    x_min, x_max = x.min(), x.max()
    y_min, y_max = y.min(), y.max()
    
    # Создаем регулярную сетку
    x_grid = np.linspace(x_min, x_max, 100)
    y_grid = np.linspace(y_min, y_max, 100)
    X, Y = np.meshgrid(x_grid, y_grid)
    
    # Выбираем параметр сглаживания, например, s = 0.1 * len(x)
    s_value = 0.1 * len(x)
    spline = SmoothBivariateSpline(x, y, z, s=s_value)
    # Обрезаем значения выше 1100
    
    # Выполняем интерполяцию на созданной сетке
    # Метод ev() позволяет вычислить значения интерполированной функции в заданных точках
    Z = spline.ev(X.ravel(), Y.ravel()).reshape(X.shape)
    Z[Z > 1100] = 1100
    # Проверка на отрицательные значения
    if check_negative_values(Z):
        print(f'Отрицательные значения присутствуют в {experiment_name}')
        neg_count, neg_percent = count_negative_values(Z)
        print(f'Количество отрицательных точек в поверхности: {neg_count}, {neg_percent}%')
    
    # Сохраняем результаты интерполяции
    spline_data[experiment_name] = (X, Y, Z)
    
    # Построение графика
    surf = ax.plot_surface(X, Y, Z, cmap="viridis", alpha=0.8, antialiased=False)
    ax.scatter(x, y, z, c="red", s=20, label="Исходные данные", depthshade=False)
    
    ax.set_xlabel("Расход топлива")
    ax.set_ylabel(y_label)
    ax.set_zlabel("ppm", rotation=90, labelpad=-6)
    ax.set_title(f'{title_on_russian(experiment_name[0])}, разбавитель - {title_on_russian(experiment_name[1])}')
    plt.legend()
    ax.view_init(elev=15, azim=55)
    ax.tick_params(axis='z', which='major', pad=-3, rotation=90)
    plt.tight_layout()
    plt.show()

# LSQBivariateSpline

In [ ]:
spline_data = {}

for experiment_name, data in experiment_data.items():
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    
    # Выбираем данные для оси y
    if 'air' in experiment_name:
        y = np.array(data.F_air)
        y_label = "Расход воздуха, кг/ч"
    else:
        y = np.array(data.F_steam)
        y_label = "Расход пара, кг"
    
    x = np.array(data.F_fuel)
    z = np.array(data.CO)
    
    # Определяем диапазоны для построения сетки
    x_min, x_max = x.min(), x.max()
    y_min, y_max = y.min(), y.max()
    
    # Создаем регулярную сетку для визуализации
    x_grid = np.linspace(x_min, x_max, 100)
    y_grid = np.linspace(y_min, y_max, 100)
    X, Y = np.meshgrid(x_grid, y_grid)
    
    # Задаем количество внутренних узлов по каждой оси
    num_knots_x = 5
    num_knots_y = 5
    # Узлы должны находиться строго между крайними значениями
    tx = np.linspace(x_min, x_max, num_knots_x + 2)[1:-1]
    ty = np.linspace(y_min, y_max, num_knots_y + 2)[1:-1]
    
    # Создаем интерполятор LSQBivariateSpline
    # bbox определяет границы данных: [x_min, x_max, y_min, y_max]
    spline = LSQBivariateSpline(x, y, z, tx, ty, bbox=[x_min, x_max, y_min, y_max], kx=3, ky=3)
    
    # Вычисляем интерполированные значения на сетке
    Z = spline.ev(X.ravel(), Y.ravel()).reshape(X.shape)

    Z[Z > 1100] = 1100
    
    # Проверка на отрицательные значения
    if check_negative_values(Z):
        print(f'Отрицательные значения присутствуют в {experiment_name}')
        neg_count, neg_percent = count_negative_values(Z)
        print(f'Количество отрицательных точек в поверхности: {neg_count}, {neg_percent}%')
    
    # Сохраняем результаты интерполяции
    spline_data[experiment_name] = (X, Y, Z)
    
    # Построение графика
    surf = ax.plot_surface(X, Y, Z, cmap="viridis", alpha=0.8, antialiased=False)
    ax.scatter(x, y, z, c="red", s=20, label="Исходные данные", depthshade=False)
    
    ax.set_xlabel("Расход топлива")
    ax.set_ylabel(y_label)
    ax.set_zlabel("ppm", rotation=90, labelpad=-6)
    ax.set_title(f'{title_on_russian(experiment_name[0])}, разбавитель - {title_on_russian(experiment_name[1])}')
    plt.legend()
    ax.view_init(elev=15, azim=55)
    ax.tick_params(axis='z', which='major', pad=-3, rotation=90)
    plt.tight_layout()
    plt.show()

In [ ]:
kriging_data = {}

for experiment_name, data in experiment_data.items():
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    
    # Выбираем данные для оси y
    if 'air' in experiment_name:
        y = np.array(data.F_air)
        y_label = "Расход воздуха, кг/ч"
    else:
        y = np.array(data.F_steam)
        y_label = "Расход пара, кг"
    
    x = np.array(data.F_fuel)
    z = np.array(data.CO)
    
    # Определяем диапазоны для построения сетки
    x_min, x_max = x.min(), x.max()
    y_min, y_max = y.min(), y.max()
    
    # Создаем регулярную сетку для визуализации
    x_grid = np.linspace(x_min, x_max, 100)
    y_grid = np.linspace(y_min, y_max, 100)
    X, Y = np.meshgrid(x_grid, y_grid)
    
    # Создаем интерполятор методом Кригинга (Ordinary Kriging)
    # variogram_model может быть, например, 'linear', 'power', 'spherical', 'gaussian' или 'exponential'
    OK = OrdinaryKriging(x, y, z, variogram_model='exponential', verbose=False, enable_plotting=False)
    
    # Выполняем интерполяцию на сетке
    # Метод execute возвращает интерполированное поле Z и оценку дисперсии ss (не используется здесь)
    Z, ss = OK.execute('grid', x_grid, y_grid)
    
    # Проверка на отрицательные значения
    if check_negative_values(Z):
        print(f'Отрицательные значения присутствуют в {experiment_name}')
        neg_count, neg_percent = count_negative_values(Z)
        print(f'Количество отрицательных точек в поверхности: {neg_count}, {neg_percent}%')
    
    # Сохраняем результаты интерполяции
    kriging_data[experiment_name] = (X, Y, Z)
    
    # Построение графика
    surf = ax.plot_surface(X, Y, Z, cmap="viridis", alpha=0.8, antialiased=False)
    ax.scatter(x, y, z, c="red", s=20, label="Исходные данные", depthshade=False)
    
    ax.set_xlabel("Расход топлива")
    ax.set_ylabel(y_label)
    ax.set_zlabel("ppm", rotation=90, labelpad=-6)
    ax.set_title(f'{title_on_russian(experiment_name[0])}, разбавитель - {title_on_russian(experiment_name[1])}')
    plt.legend()
    ax.view_init(elev=15, azim=55)
    ax.tick_params(axis='z', which='major', pad=-3, rotation=90)
    plt.tight_layout()
    plt.show()

# Дифференциальные поверхности

In [ ]:
def calculate_threshold_percentage(grid_data, threshold):
    """
    Считает и выводит процент точек на поверхности, превышающих заданный порог.

    Параметры:
        grid_data (dict): Результаты интерполяции из plot_grid_interpolation
        threshold (float): Пороговое значение CO для определения аномалий
    """
    for exp_name, (X, Y, Z) in grid_data.items():
        # Рассчитываем общее количество точек
        total_points = Z.size
        
        # Считаем точки выше порога
        above_threshold = np.sum(Z > threshold)
        
        # Рассчитываем долю в процентах
        percent_above = (above_threshold / total_points) * 100
        
        # Форматируем вывод
        print(f"Эксперимент '{exp_name}':")
        print(f"Точек выше порога {threshold} ppm: {percent_above:.2f}%")
        print(f"Всего проанализировано точек: {total_points}\n")


def plot_grid_interpolation(method):
    """
    Строит контурные графики дифференциальной поверхности для экспериментальных данных.
    
    Параметры:
      method : str, способ интерполяции ('linear', 'nearest', 'cubic')
    """
    grid_data = {}
    shift = 5  # Шаг для вычисления дифференциальной поверхности
    
    for experiment_name, data in experiment_data.items():
        plt.figure(figsize=(10, 6))
        ax = plt.gca()
        
        # Выбираем данные для оси y
        if 'air' in experiment_name:
            y = np.array(data.F_air)
            y_label = "Расход воздуха, кг/ч"
        else:
            y = np.array(data.F_steam)
            y_label = "Расход пара, кг"
        
        x = np.array(data.F_fuel)
        z = np.array(data.CO)
        
        # Диапазоны для сетки
        x_min, x_max = x.min(), x.max()
        y_min, y_max = y.min(), y.max()
        
        # Создаем регулярную сетку
        x_grid = np.linspace(x_min, x_max, 100)
        y_grid = np.linspace(y_min, y_max, 100)
        X, Y = np.meshgrid(x_grid, y_grid)
        
        # Интерполяция
        Z = griddata((x, y), z, (X, Y), method=method)
        
        # Вычисляем дифференциальную поверхность
        Z_diff = np.abs(Z[:, shift:] - Z[:, :-shift])
        x_grid_diff = x_grid[shift:]
        X_diff, Y_diff = np.meshgrid(x_grid_diff, y_grid)
        
        # Сохраняем результаты
        grid_data[experiment_name] = (X_diff, Y_diff, Z_diff)
        
        # Построение контурного графика
        contour = ax.contourf(
            X_diff, Y_diff, Z_diff,
            cmap="viridis",
            levels=20,
            alpha=0.8
        )
        
        # Настройка цветовой шкалы
        cbar = plt.colorbar(contour, ax=ax)
        cbar.set_label('ΔCO, ppm', rotation=270, labelpad=15)
        
        # Настройка осей и заголовка
        ax.set_xlabel("Расход топлива", labelpad=10)
        ax.set_ylabel(y_label, labelpad=10)
        ax.set_title(f'{title_on_russian(experiment_name[0])}, разбавитель - {title_on_russian(experiment_name[1])}')
        
        # Оптимизация размещения элементов
        plt.tight_layout()
        plt.show()

    return grid_data

In [ ]:
# Сначала получаем данные
grid_results = plot_grid_interpolation(method='linear')

# Затем анализируем порог
calculate_threshold_percentage(grid_results, threshold=75)

In [ ]:
# Сначала получаем данные
grid_results = plot_grid_interpolation(method='linear')

# Затем анализируем порог
calculate_threshold_percentage(grid_results, threshold=75)

In [ ]:
def delete_underline(string_with_underline: str) -> tuple:
    parts = string_with_underline.split('_')
    
    # Ищем индекс "steam" или "air"
    if "steam" in parts:
        idx = parts.index("steam")
    elif "air" in parts:
        idx = parts.index("air")
    else:
        raise ValueError("В строке нет ни 'steam', ни 'air'")
    
    # Топливо может состоять из нескольких слов через '_'
    fuel_name = '_'.join(parts[:idx])
    return (fuel_name, parts[idx], parts[idx + 1])

def get_fuels_combinations(CO=True) -> list[FuelCombination]:
    if CO:
        fuel_combinations = [
            'diesel_steam_CO',
            'diesel_air_CO',
            'crude_oil_steam_CO',
            'heavy_oil_steam_CO',
            'heavy_oil_air_CO',
            'kerosene_steam_CO',
            'kerosene_air_CO',
            'waste_oil_steam_CO',
            'waste_oil_40_2024_steam_CO',
            'waste_oil_60_2024_steam_CO'
        ]
    else:
        fuel_combinations = [
            'diesel_steam_O2',
            'crude_oil_steam_O2',
            'heavy_oil_steam_O2',
            'kerosene_steam_O2',
            'waste_oil_steam_O2',\
        ]
    return fuel_combinations

"""АННОТАЦИЯ ТИПОВ"""
FuelCombination: TypeAlias = str

class ApproxData(NamedTuple):
    """Аппроксимированная поверхность со значением расхода топлива и вводимого компонента"""
    fuel_consumption: np.ndarray
    additive_consumption: np.ndarray
    approximated_surface: np.ndarray

class DiffData(NamedTuple):
    """Дифференциальная поверхность со значением расхода топлива и вводимого компонента"""
    fuel_consumption: np.ndarray
    additive_consumption: np.ndarray
    diff_surface: np.ndarray

class MinPoints(NamedTuple):
    """Координаты минимимальных значений каждой строки в матрице"""
    x: np.ndarray
    y: np.ndarray

class LinearCoeffs(NamedTuple):
    """Коэффициенты линейной прямой"""
    angular_coefficient: float  # Угловой коэффицент
    b: float

class LevelPoints(NamedTuple):
    """Координаты выбранного уровне contourf"""
    x: np.ndarray
    y: np.ndarray

"""ОШИБКИ"""

def fuel_combination_key_error() -> KeyError:
    return KeyError('Сочетание топлива, добавки и компонента не найдено в словаре!')

def get_rbf_data(CO=True) -> Mapping[FuelCombination, ApproxData]:
    fuel_combinations = get_fuels_combinations(CO=CO)
    rbf_data = {}
    
    for fuel_combination in fuel_combinations:
        ex = ExperimentData()
        ex.get_experiment_data(*delete_underline(fuel_combination))
        rbf_data[fuel_combination] = ex.get_rbf_data()
    return rbf_data

def calculate_threshold_percentage_rbf(rbf_data, threshold):
    """
    Считает и выводит процент точек на поверхности RBF-интерполяции, превышающих заданный порог.
    
    Параметры:
        rbf_data (dict): Результаты RBF-интерполяции (словарь с кортежами (X, Y, Z))
        threshold (float): Пороговое значение CO для определения аномалий
    """
    for exp_name, (X, Y, Z) in rbf_data.items():
        # Рассчитываем общее количество точек
        total_points = Z.size
        
        # Считаем точки выше порога
        above_threshold = np.sum(Z > threshold)
        
        # Рассчитываем долю в процентах
        percent_above = (above_threshold / total_points) * 100
        
        # Форматируем вывод
        print(f"Эксперимент '{exp_name}':")
        print(f"Точек выше порога {threshold} ppm: {percent_above:.2f}%")
        print(f"Всего проанализировано точек: {total_points}\n")


def plot_rbf_interpolation(CO=True, shift: int = 5, levels: int = 5):
    """
    Строит контурные графики дифференциальной поверхности для экспериментальных данных
    на основе готовых данных RBF (как в get_diff_data).
    
    Параметры:
        CO : bool, использовать ли CO (по умолчанию True)
        shift : int, шаг для вычисления дифференциальной поверхности по оси X
        levels : int, количество уровней для contourf
    """
    rbf_data = get_rbf_data(CO=CO)  # <-- используем готовые RBF-данные
    diff_data = {}

    for fuel_combination in rbf_data:
        fuel_cons, add_cons, surf = rbf_data[fuel_combination]

        # --- Дифференциальная поверхность (точно как в get_diff_data) ---
        diff = np.abs(surf[:, 0:surf.shape[1]-shift] - surf[:, shift:surf.shape[1]])
        fuel_cons_diff = fuel_cons[shift:]
        X_diff, Y_diff = np.meshgrid(fuel_cons_diff, add_cons)
        # ----------------------------------------------------------------

        diff_data[fuel_combination] = (X_diff, Y_diff, diff)

        # Построение графика
        fig, ax = plt.subplots(figsize=(10, 6))
        contour = ax.contourf(
            X_diff, Y_diff, diff,
            cmap="viridis",
            levels=levels
        )

        # Цветовая шкала
        cbar = plt.colorbar(contour, ax=ax)
        cbar.set_label('ΔCO, ppm', rotation=270, labelpad=15)

        # Подписи и заголовки
        ax.set_xlabel("Расход топлива", labelpad=10)
        if "air" in fuel_combination:
            ax.set_ylabel("Расход воздуха, кг/ч", labelpad=10)
        else:
            ax.set_ylabel("Расход пара, кг/ч", labelpad=10)

        fuel_name, diluent, component = delete_underline(fuel_combination)

        ax.set_title(f'{title_on_russian(fuel_name)}, разбавитель - {title_on_russian(diluent)}')

        plt.tight_layout()
        plt.show()

    return diff_data


In [ ]:
rbf_data = plot_rbf_interpolation()

calculate_threshold_percentage_rbf(rbf_data, 75)